In [4]:
# ============================================================
# ONE CELL (TOX171, multiclass): OPTUNA with GO-LR ONCE per trial 
# Per trial:
#   1) GO-LR fit on FULL X -> Pi_star (fixed for all folds in that trial)
#   2) 5x5 CV folds:
#        NSC configure on train only (uses fixed Pi_star)
#        compress train/val
#        TabPFN fit on train, eval on val
# ============================================================

import os, sys, gc, random, warnings
warnings.filterwarnings("ignore")

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")

import optuna
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from tqdm import tqdm

#   GraphFeatureOrdering, pidf_segpca, TabPFN25Head, TabPFN25Config
from gotabpfn import GraphFeatureOrdering, pidf_segpca, TabPFN25Head, TabPFN25Config
PIDFSegPCA = pidf_segpca  # alias to match your earlier naming

# -----------------------
# Config
# -----------------------
SEED = 42
DATA_FILE  = "TOX171_combined_encoded.csv"
TARGET_COL = "Label"

N_TRIALS = 150
GPU_ID = 7  # change if needed

# If you want to force which GPU is visible for kmeans_gpu + tabpfn:
if "torch" not in sys.modules:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)

# -----------------------
# Utils
# -----------------------
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass
        torch.cuda.empty_cache()

def ensure_multiclass_contiguous(y: np.ndarray):
    y = np.asarray(y).reshape(-1)
    uniq = np.unique(y)
    uniq_sorted = np.sort(uniq)
    class_map = {orig: i for i, orig in enumerate(uniq_sorted.tolist())}
    y_enc = np.vectorize(class_map.get, otypes=[np.int64])(y).astype(np.int64)
    return y_enc, class_map, int(len(class_map))

def compute_deltas_adjacent_corr(X_tr: np.ndarray, Pi_star: list[int], eps: float = 1e-12) -> torch.Tensor:
    """
    delta[t] = 1 - |corr(feature_{Pi[t]}, feature_{Pi[t+1]})|
    Returns torch.Tensor shape (m-1,) on CPU.
    """
    X = torch.from_numpy(X_tr).float()  # CPU
    perm = torch.tensor(Pi_star, dtype=torch.long)
    Xp = X[:, perm]
    Xc = Xp - Xp.mean(dim=0, keepdim=True)
    std = Xc.std(dim=0, unbiased=False, keepdim=True).clamp_min(eps)
    Z = Xc / std
    corr = (Z[:, :-1] * Z[:, 1:]).mean(dim=0)
    return (1.0 - corr.abs()).cpu()

# -----------------------
# Load data
# -----------------------
seed_everything(SEED)
df = pd.read_csv(DATA_FILE)

y_raw = df[TARGET_COL].to_numpy()
X_df  = df.drop(columns=[TARGET_COL])

y, class_map, NUM_CLASSES = ensure_multiclass_contiguous(y_raw)

# Standardize globally (leakage OK in your workflow)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df.values).astype(np.float32, copy=False)

X_all = np.asarray(X_scaled, dtype=np.float32, order="C")
y_all = np.asarray(y, dtype=np.int64)

print(f"[DATA] {DATA_FILE} | X={X_all.shape} | C={NUM_CLASSES} | map={class_map}")
print("[GPU ] cuda=", torch.cuda.is_available(), "| visible_gpus=", torch.cuda.device_count())

# 5x5 CV
rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)

# Devices
NSC_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TABPFN_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------
# Optuna objective
# -----------------------
def objective(trial: optuna.Trial):
    seed_everything(SEED)

    # --------- Tune GO-LR (once per trial) ----------
    go_metric = trial.suggest_categorical(
        "go_metric", ["correlation", "cosine", "manhattan", "euclidean", "kl_divergence"]
    )
    go_k = trial.suggest_int("go_num_clusters", 4, 12)
    go_refine_passes = trial.suggest_int("go_refine_passes", 1, 3)
    go_direction = trial.suggest_categorical("go_direction_select", [True, False])

    # IMPORTANT: prefer GPU KMeans first; CPU only as fallback
    # So we DO NOT tune go_use_cpu_kmeans here.
    # We try use_cpu_kmeans=False first, then fallback True if needed.

    # --------- Tune NSC ----------
    nsc_seg = trial.suggest_categorical("nsc_segmentation", ["uniform", "largest_jump", "equal_mass"])
    nsc_m_rule = trial.suggest_categorical("nsc_m_rule", ["default", "idf", "gamma"])
    nsc_tau = trial.suggest_categorical("nsc_tau", [0.95, 0.99, 0.9975])
    nsc_gamma = trial.suggest_float("nsc_gamma", 1.0, 3.0)
    nsc_beta = trial.suggest_float("nsc_beta", 0.0, 0.9)
    nsc_Mmin = trial.suggest_categorical("nsc_Mmin", [16, 32, 48, 64])
    nsc_Mmax = trial.suggest_categorical("nsc_Mmax", [128, 256, 384, 512, 640])
    nsc_lmin = trial.suggest_categorical("nsc_lmin", [8, 12, 16])
    assume_standardized = trial.suggest_categorical("assume_standardized", [True, False])

    # TabPFN seed (head is frozen; but its internal rng can matter)
    tabpfn_seed = trial.suggest_categorical("tabpfn_seed", [0, 1, 2, 3, 4, 42])

    # ---- GO-LR ONCE per trial ----
    go = GraphFeatureOrdering(
        num_clusters=int(go_k),
        metric=go_metric,
        refine=True,
        direction_select=bool(go_direction),
        refine_passes=int(go_refine_passes),
    )

    try:
        # Prefer GPU kmeans
        Pi_star, _, _, _ = go.fit(X_all, seed=SEED, deterministic=True, use_cpu_kmeans=False)
    except RuntimeError as e:
        # Fallback to CPU kmeans if GPU path OOMs/fails
        cleanup_cuda()
        try:
            Pi_star, _, _, _ = go.fit(X_all, seed=SEED, deterministic=True, use_cpu_kmeans=True)
        except Exception:
            raise optuna.TrialPruned(f"GO-LR failed (metric={go_metric})")

    # ---- 5x5 CV: NSC+TabPFN per fold using fixed Pi_star ----
    head_cfg = TabPFN25Config(
        task_type="multiclass",
        num_classes=int(NUM_CLASSES),
        device=TABPFN_DEVICE,
        random_state=int(tabpfn_seed),
    )

    accs = []
    for fold_id, (tr_idx, va_idx) in enumerate(rkf.split(X_all, y_all), start=1):
        X_tr = X_all[tr_idx]
        y_tr = y_all[tr_idx]
        X_va = X_all[va_idx]
        y_va = y_all[va_idx]

        # NSC config on TRAIN only
        nsc = PIDFSegPCA(
            segmentation=nsc_seg,
            l_min=int(nsc_lmin),
            m_rule=nsc_m_rule,
            gamma=float(nsc_gamma),
            beta=float(nsc_beta),
            tau=float(nsc_tau),
            M_min=int(nsc_Mmin),
            M_max=int(nsc_Mmax),
            assume_standardized=bool(assume_standardized),
            device=NSC_DEVICE,
        )

        deltas = None if nsc_seg == "uniform" else compute_deltas_adjacent_corr(X_tr, Pi_star)

        X_tr_t = torch.from_numpy(X_tr)
        nsc.configure(Pi_star=Pi_star, X_train=X_tr_t, tau=float(nsc_tau), deltas=deltas)

        Z_tr = nsc.compress(X_tr_t, mode="flatten").cpu().numpy()
        Z_va = nsc.compress(torch.from_numpy(X_va), mode="flatten").cpu().numpy()

        # TabPFN fit/eval
        head = TabPFN25Head(head_cfg)
        head.fit(Z_tr, y_tr)

        P = head.predict_proba(Z_va)
        pred = np.argmax(P, axis=1)
        acc = float(accuracy_score(y_va, pred))
        accs.append(acc)

        # prune on running mean
        trial.report(float(np.mean(accs)), step=fold_id)
        if trial.should_prune():
            cleanup_cuda()
            raise optuna.TrialPruned()

        cleanup_cuda()

    mean_acc = float(np.mean(accs))
    std_acc  = float(np.std(accs, ddof=1))
    trial.set_user_attr("mean_acc", mean_acc)
    trial.set_user_attr("std_acc", std_acc)
    return mean_acc

# -----------------------
# Run Optuna
# -----------------------
sampler = optuna.samplers.TPESampler(seed=SEED, multivariate=True, group=True)
pruner  = optuna.pruners.MedianPruner(n_warmup_steps=10)  # warmup because 25 folds

study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True, gc_after_trial=True, n_jobs=1)

best = study.best_trial
print("\n================ BEST TRIAL ================")
print(f"mean_acc ± std_acc = {best.user_attrs.get('mean_acc', best.value):.6f} ± {best.user_attrs.get('std_acc', float('nan')):.6f}")
print("\nBest hyperparameters:")
for k, v in best.params.items():
    print(f"  {k}: {v}")

[I 2026-01-21 20:08:17,456] A new study created in memory with name: no-name-5d6d71fa-de0b-46d6-a0fb-0e3d0033fc0a


[DATA] TOX171_combined_encoded.csv | X=(171, 5748) | C=4 | map={0: 0, 1: 1, 2: 2, 3: 3}
[GPU ] cuda= True | visible_gpus= 8


  0%|          | 0/150 [00:00<?, ?it/s]

[I 2026-01-21 20:10:09,797] Trial 0 finished with value: 0.8947226890756304 and parameters: {'go_metric': 'cosine', 'go_num_clusters': 5, 'go_refine_passes': 1, 'go_direction_select': True, 'nsc_segmentation': 'equal_mass', 'nsc_m_rule': 'default', 'nsc_tau': 0.9975, 'nsc_gamma': 1.8638900372842315, 'nsc_beta': 0.26210622617823776, 'nsc_Mmin': 16, 'nsc_Mmax': 256, 'nsc_lmin': 12, 'assume_standardized': False, 'tabpfn_seed': 0}. Best is trial 0 with value: 0.8947226890756304.
[I 2026-01-21 20:12:08,702] Trial 1 finished with value: 0.8550924369747898 and parameters: {'go_metric': 'euclidean', 'go_num_clusters': 9, 'go_refine_passes': 1, 'go_direction_select': False, 'nsc_segmentation': 'largest_jump', 'nsc_m_rule': 'default', 'nsc_tau': 0.95, 'nsc_gamma': 1.0904545778210761, 'nsc_beta': 0.2927972976869379, 'nsc_Mmin': 48, 'nsc_Mmax': 512, 'nsc_lmin': 8, 'assume_standardized': False, 'tabpfn_seed': 2}. Best is trial 0 with value: 0.8947226890756304.
[I 2026-01-21 20:14:33,552] Trial 2 fi